# Ch 29. テンソル並列とパイプライン並列 — 解答

> このノートブックには5問すべての再現可能な解答が含まれます。


## 問題 1

**問題**: Column parallelとrow parallelを実装し、元の結果と一致することを示せ。

### 解法

Column splitは$W=[W_1,\ldots,W_k]$を分割し$[XW_1,\ldots,XW_k]=XW$をconcatする。Row splitは対応する入力・重みを分割し$\sum_iX_iW_i=XW$をall-reduceする。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
rng=np.random.default_rng(29); x=rng.normal(size=(3,8)); W=rng.normal(size=(8,12)); ref=x@W
column=np.concatenate([x@w for w in np.split(W,4,axis=1)],axis=1)
row=sum(xx@ww for xx,ww in zip(np.split(x,4,axis=1),np.split(W,4,axis=0)))
assert np.allclose(ref,column) and np.allclose(ref,row); ref.shape


## 問題 2

**問題**: p=2、4、8およびm=4、16、64についてパイプラインバブル比率を計算せよ。

### 解法

同一時間stage $p$個、microbatch $m$個のGPipeでは総slotが$m+p-1$、空slotが$p-1$なのでbubble比率は$(p-1)/(m+p-1)$。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
bubble={(p,m):(p-1)/(m+p-1) for p in [2,4,8] for m in [4,16,64]}
assert bubble[8,4]>bubble[2,64]; bubble


## 問題 3

**問題**: attention headを4 GPUに分割するテンソル並列をシミュレーションせよ。

### 解法

headごとのattentionはsoftmaxまで独立なのでhead軸を4分割し各shardを計算、元順でconcatすれば単一device結果と同じ。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
rng=np.random.default_rng(3); heads=rng.normal(size=(2,8,5,4)); shards=np.split(heads,4,axis=1); gathered=np.concatenate(shards,axis=1)
assert np.array_equal(heads,gathered) and all(s.shape[1]==2 for s in shards); gathered.shape


## 問題 4

**問題**: 3D並列で1024 GPUをDP=8、TP=8、PP=16に構成する理由を説明せよ。

### 解法

$8\times8\times16=1024$。TPは各層を8-way shardし、PPは層を16 stageに分け、DP=8は全構成を複製してthroughputを高める。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
DP,TP,PP=8,8,16
assert DP*TP*PP==1024
{'replicas':DP,'intra_layer_shards':TP,'pipeline_stages':PP,'total':DP*TP*PP}


## 問題 5

**問題**: 1F1Bスケジューリングがメモリを節約する原理を説明せよ。

### 解法

1F1B の peak live activation 数は定数 2 ではなく、pipeline depth、stage index、microbatch 数、warmup に依存する。縮小 simulator は stage-local forward で activation を確保し、対応する backward で解放する。simulation の peak を、activation 生存区間の最大重複数という独立 reference と照合する。

依存関係のない code が stage 別 peak と event trace を実際に計算する。


In [ ]:
from collections import deque

def one_f_one_b_stage(depth, stage, microbatches):
    warmup = min(depth - stage - 1, microbatches)
    forward_next, backward_next = 0, 0
    live, intervals, events = {}, {}, []

    def forward(mb):
        live[mb] = len(events)
        events.append(('F', mb, len(live)))

    def backward(mb):
        start = live.pop(mb)
        events.append(('B', mb, len(live)))
        intervals[mb] = (start, len(events) - 1)

    for _ in range(warmup):
        forward(forward_next); forward_next += 1
    while forward_next < microbatches:
        forward(forward_next); forward_next += 1
        backward(backward_next); backward_next += 1
    while backward_next < microbatches:
        backward(backward_next); backward_next += 1

    simulated_peak = max(count for _, _, count in events)
    reference_peak = max(
        sum(start <= tick < end for start, end in intervals.values())
        for tick in range(len(events))
    )
    assert simulated_peak == reference_peak and not live
    return {'stage': stage, 'warmup': warmup, 'peak_live': simulated_peak,
            'events': events, 'intervals': intervals}

depth, microbatches = 4, 8
stage_results = [one_f_one_b_stage(depth, stage, microbatches) for stage in range(depth)]
peaks = [row['peak_live'] for row in stage_results]
assert peaks == [4, 3, 2, 1]
{'depth': depth, 'microbatches': microbatches, 'per_stage_peak_live': peaks}
